# 03 — Model Experiments

Training Ridge Regression, Random Forest, XGBoost, and LSTM models.

**Evaluation metrics:** MAE, RMSE, R² per horizon (24h, 48h, 72h)  
**Selection criteria:** Performance vs complexity tradeoff

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Load and Prepare Data

In [ ]:
# Load feature-engineered splits
train_feat = pd.read_csv('../data/processed/train_features.csv')
train_tgt = pd.read_csv('../data/processed/train_targets.csv')
val_feat = pd.read_csv('../data/processed/val_features.csv')
val_tgt = pd.read_csv('../data/processed/val_targets.csv')
test_feat = pd.read_csv('../data/processed/test_features.csv')
test_tgt = pd.read_csv('../data/processed/test_targets.csv')

target_cols = ['target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']

# Select numeric feature columns only
exclude = ['timestamp', 'location_id', 'city_name', 'data_source', 'aqi_category',
           'aqi_standard', 'aqi_method', 'aqi_method_version', 'aqi_source']
feature_cols = [c for c in train_feat.columns if c not in exclude
                and train_feat[c].dtype in ['float64', 'int64', 'bool']]

print(f"Features: {len(feature_cols)}")
print(f"Train: {len(train_feat):,} rows")
print(f"Val:   {len(val_feat):,} rows")
print(f"Test:  {len(test_feat):,} rows")

In [ ]:
# Prepare X and y, drop rows with NaN targets
def prepare(feat, tgt, feature_cols, target_cols):
    mask = tgt[target_cols].notna().all(axis=1)
    X = feat.loc[mask, feature_cols].fillna(0).values
    y = tgt.loc[mask, target_cols].values
    return X, y

X_train, y_train = prepare(train_feat, train_tgt, feature_cols, target_cols)
X_val, y_val = prepare(val_feat, val_tgt, feature_cols, target_cols)
X_test, y_test = prepare(test_feat, test_tgt, feature_cols, target_cols)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

## 2. Evaluation Helper

In [ ]:
def evaluate(y_true, y_pred, model_name):
    """Evaluate multi-output predictions per horizon."""
    results = []
    for i, h in enumerate([24, 48, 72]):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        results.append({'Model': model_name, 'Horizon': f'{h}h',
                        'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R²': round(r2, 4)})
    # Overall
    mae_all = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    rmse_all = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
    r2_all = r2_score(y_true.flatten(), y_pred.flatten())
    results.append({'Model': model_name, 'Horizon': 'All',
                    'MAE': round(mae_all, 2), 'RMSE': round(rmse_all, 2), 'R²': round(r2_all, 4)})
    return pd.DataFrame(results)

## 3. Ridge Regression (Baseline)

In [ ]:
t0 = time.time()
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)
y_pred_ridge = ridge.predict(X_test_s)
ridge_time = time.time() - t0

ridge_results = evaluate(y_test, y_pred_ridge, 'Ridge')
ridge_results['Train Time (s)'] = round(ridge_time, 3)
print(ridge_results.to_string(index=False))

## 4. Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=100, max_depth=20, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_time = time.time() - t0

rf_results = evaluate(y_test, y_pred_rf, 'RandomForest')
rf_results['Train Time (s)'] = round(rf_time, 1)
print(rf_results.to_string(index=False))

## 5. XGBoost

In [ ]:
t0 = time.time()
xgb_model = MultiOutputRegressor(
    xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, verbosity=0
    )
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
xgb_time = time.time() - t0

xgb_results = evaluate(y_test, y_pred_xgb, 'XGBoost')
xgb_results['Train Time (s)'] = round(xgb_time, 1)
print(xgb_results.to_string(index=False))

## 6. Results Comparison

In [ ]:
all_results = pd.concat([ridge_results, rf_results, xgb_results], ignore_index=True)
print(all_results.to_string(index=False))

In [ ]:
# MAE comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
horizons = ['24h', '48h', '72h']
models = ['Ridge', 'RandomForest', 'XGBoost']
colors = ['#2196F3', '#FF5722', '#4CAF50']

for ax, h in zip(axes, horizons):
    h_data = all_results[all_results['Horizon'] == h]
    bars = ax.bar(h_data['Model'], h_data['MAE'], color=colors, edgecolor='white')
    ax.set_title(f'MAE — {h}')
    ax.set_ylabel('MAE')
    for bar, val in zip(bars, h_data['MAE']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison: MAE per Horizon', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# R² comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, h in zip(axes, horizons):
    h_data = all_results[all_results['Horizon'] == h]
    bars = ax.bar(h_data['Model'], h_data['R²'], color=colors, edgecolor='white')
    ax.set_title(f'R² — {h}')
    ax.set_ylabel('R²')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, h_data['R²']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison: R² per Horizon', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Training time comparison
fig, ax = plt.subplots(figsize=(8, 4))
time_data = all_results[all_results['Horizon'] == '24h'][['Model', 'Train Time (s)']]
bars = ax.bar(time_data['Model'], time_data['Train Time (s)'], color=colors, edgecolor='white')
ax.set_title('Training Time')
ax.set_ylabel('Seconds')
for bar, val in zip(bars, time_data['Train Time (s)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}s', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()